## Variables climaticas de Tecate 

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import requests

#  Ubicamos la raíz del proyecto para usar las rutas definidas en src/auxiliar_functions.py
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.auxiliar_functions import RAW_DATA_DIR, PROCESSED_DATA_DIR

# Coordenadas de la planta en Tecate, Baja California, México (pág. 10 de la consigna)
TECATE_LAT = 32.5722
TECATE_LON = -116.6283
COL_TARGET = "Frio (Kw)_Consolidado EE"



# Función reproducible para obtener clima diario + radiación solar (Open-Meteo)

def obtener_clima_tecate(
    fecha_inicio: str,
    fecha_fin: str,
    ruta_cache: Path = RAW_DATA_DIR / "clima_tecate_openmeteo.csv",
) -> pd.DataFrame:
    """
    Descarga desde Open-Meteo (o lee de caché local) las temperaturas diarias
    y la radiación solar acumulada de Tecate para el rango [fecha_inicio, fecha_fin].
    """
    req_ini = pd.to_datetime(fecha_inicio)
    req_fin = pd.to_datetime(fecha_fin)
    columnas_esperadas = [
        "DIA",
        "temp_max_c",
        "temp_min_c",
        "temp_media_c",
        "temp_aparente_c",
        "radiacion_mj_m2", #basicamente mide la energia solar que hay en ese dia, es como decir, el sol que pega en la planta y que tanta energia absorve (por ejemplo se calientan las tuberias)
    ]

    # Si el CSV local ya tiene todas las columnas (incluida radiación) y cubre las fechas, se reutiliza
    if ruta_cache.exists():
        df_cache = pd.read_csv(ruta_cache, parse_dates=["DIA"])
        if (
            set(columnas_esperadas).issubset(df_cache.columns)
            and df_cache["DIA"].min() <= req_ini
            and df_cache["DIA"].max() >= req_fin
        ):
            mask = df_cache["DIA"].between(req_ini, req_fin)
            return df_cache.loc[mask, columnas_esperadas].sort_values("DIA").reset_index(drop=True)

    # Consulta a la API histórica de Open-Meteo
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": TECATE_LAT,
        "longitude": TECATE_LON,
        "start_date": req_ini.strftime("%Y-%m-%d"),
        "end_date": req_fin.strftime("%Y-%m-%d"),
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "temperature_2m_mean",
            "apparent_temperature_mean",
            "shortwave_radiation_sum",
        ],
        "timezone": "America/Tijuana",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    daily = resp.json()["daily"]

    df_clima = pd.DataFrame({
        "DIA": pd.to_datetime(daily["time"]),
        "temp_max_c": daily["temperature_2m_max"],
        "temp_min_c": daily["temperature_2m_min"],
        "temp_media_c": daily["temperature_2m_mean"],
        "temp_aparente_c": daily["apparent_temperature_mean"],
        "radiacion_mj_m2": daily["shortwave_radiation_sum"],
    }).sort_values("DIA").reset_index(drop=True)

    # Guardamos en caché con salto de línea '\n' para mantener el mismo formato en Windows y Linux
    df_clima.to_csv(ruta_cache, index=False, lineterminator="\n")
    return df_clima



# Creación de variables derivadas del clima (Sección 2.2: lags y medias móviles)
def agregar_features_clima(df_clima: pd.DataFrame) -> pd.DataFrame:
    df_c = df_clima.sort_values("DIA").copy()

    # Amplitud térmica dentro del mismo día D
    df_c["amplitud_termica_c"] = df_c["temp_max_c"] - df_c["temp_min_c"]

    # Variables de lag (día anterior D-1) y promedios móviles (últimos 3 y 7 días hasta D)
    df_c["temp_media_lag1"] = df_c["temp_media_c"].shift(1)
    df_c["temp_media_roll3"] = df_c["temp_media_c"].rolling(window=3, min_periods=1).mean()
    df_c["temp_media_roll7"] = df_c["temp_media_c"].rolling(window=7, min_periods=1).mean()
    df_c["temp_max_roll3"] = df_c["temp_max_c"].rolling(window=3, min_periods=1).mean()
    df_c["radiacion_roll3"] = df_c["radiacion_mj_m2"].rolling(window=3, min_periods=1).mean()
    df_c["radiacion_roll7"] = df_c["radiacion_mj_m2"].rolling(window=7, min_periods=1).mean()

    return df_c


# Carga del dataset unificado (dataset_trabajo.csv), cierre 23:59 y target D+1

ruta_trabajo = PROCESSED_DATA_DIR / "dataset_trabajo.csv"
df_horario_trabajo = pd.read_csv(ruta_trabajo, parse_dates=["DIA"], low_memory=False)

# Paso 1 de la consigna: filtrar el cierre diario de las 23:59
df_diario = (
    df_horario_trabajo[df_horario_trabajo["HORA"] == "23:59"]
    .sort_values("DIA")
    .reset_index(drop=True)
    .copy()
)

# Paso 2 de la consigna: construir target y = Frio(D+1) con shift(-1) solo cuando la fila siguiente es D+1
es_dia_consecutivo = (df_diario["DIA"].shift(-1) - df_diario["DIA"]).dt.days == 1
df_diario["Frio_manana_y"] = np.where(
    es_dia_consecutivo,
    df_diario[COL_TARGET].shift(-1),
    np.nan,
)


# Unión con el clima de Tecate y evaluación de correlaciones (D y D+1)

df_clima_full = agregar_features_clima(
    obtener_clima_tecate(df_diario["DIA"].min(), df_diario["DIA"].max())
)
df_diario_clima = df_diario[["DIA", COL_TARGET, "Frio_manana_y"]].merge(
    df_clima_full, on="DIA", how="left"
)

cols_clima = [
    "radiacion_mj_m2",
    "radiacion_roll3",
    "radiacion_roll7",
    "temp_aparente_c",
    "temp_media_roll7",
    "temp_media_roll3",
    "temp_min_c",
    "temp_media_c",
    "temp_media_lag1",
    "temp_max_roll3",
    "temp_max_c",
    "amplitud_termica_c",
]

resumen_corr_clima = pd.DataFrame({
    "r_Pearson_Frio_Hoy(D)": [df_diario_clima[c].corr(df_diario_clima[COL_TARGET]) for c in cols_clima],
    "r_Pearson_Target(D+1)": [df_diario_clima[c].corr(df_diario_clima["Frio_manana_y"]) for c in cols_clima],
    "r_Spearman_Target(D+1)": [df_diario_clima[c].corr(df_diario_clima["Frio_manana_y"], method="spearman") for c in cols_clima],
}, index=cols_clima).sort_values("r_Pearson_Target(D+1)", ascending=False).round(3)

print(f"Dataset diario cargado desde dataset_trabajo.csv: {len(df_diario)} días válidos ({df_diario['DIA'].min().date()} a {df_diario['DIA'].max().date()})")
print(f"Pares consecutivos válidos (D -> D+1): {df_diario_clima['Frio_manana_y'].notna().sum()} de {len(df_diario_clima)}\n")
print("CORRELACIÓN DE VARIABLES CLIMÁTICAS DE TECATE CON FRÍO (HOY D Y MAÑANA D+1):")
print(resumen_corr_clima.to_string())

Dataset diario cargado desde dataset_trabajo.csv: 1172 días válidos (2020-07-01 a 2023-10-25)
Pares consecutivos válidos (D -> D+1): 1140 de 1172

CORRELACIÓN DE VARIABLES CLIMÁTICAS DE TECATE CON FRÍO (HOY D Y MAÑANA D+1):
                    r_Pearson_Frio_Hoy(D)  r_Pearson_Target(D+1)  r_Spearman_Target(D+1)
radiacion_roll7                     0.738                  0.742                   0.743
radiacion_roll3                     0.713                  0.719                   0.722
radiacion_mj_m2                     0.666                  0.670                   0.681
temp_aparente_c                     0.582                  0.576                   0.593
temp_media_roll7                    0.570                  0.560                   0.579
temp_media_roll3                    0.561                  0.551                   0.571
temp_min_c                          0.555                  0.551                   0.574
temp_media_c                        0.553                  0.546

- **radiacion_roll7** es la mejor variable climatica con r=0,742 y Spearman=0,743: Al promediar los ultimos 7 dias de radiacion solar, la correlacion sube 0,072 respecto al dato de un solo dia. Esto pasa pq la radiacion diaria aislada tiene ruido por dias nublados por ejemplo, el promedio semanal limpia ese ruido.
    - Estabilidad frente al dia siguiente (D+1): en las tres variables de radiacion, la correlacion con el consumo de D+1 es practicamente igual, esto confirma la capacidad predictiva verdadera que tiene esta variable utilizando solo el clima observado hasta el cierre del dia D, sin necesidad de mirar el clima futuro.

- Se descarta amplitud_termica_c y quedan como candidades principales **radiacion_roll7, radiacion_mj_m2, temp_aparente_c, temp_media_roll7 y temp_min_c** (evitamos meter variables redundantes y colineales como roll3 o max_roll3 si no aportan mejora sobre roll7)

## Variables autorregresvias (lags y medias moviles)

In [3]:
# Reindexamos Frio (Kw) sobre el calendario diario continuo (sin saltear fechas faltantes).
# Así garantizamos que lag_1 siempre sea exactamente ayer (D-1), lag_7 sea hace 7 días (D-7)
# y rolling(7) evalúe realmente los últimos 7 días calendario (sin mezclar fechas distantes).
rango_calendario = pd.date_range(df_diario["DIA"].min(), df_diario["DIA"].max(), freq="D")
serie_frio_cal = df_diario.set_index("DIA")[COL_TARGET].reindex(rango_calendario)

# Construcción de variables autorregresivas (solo usan datos <= D)
df_lags_frio = pd.DataFrame(index=rango_calendario)
df_lags_frio.index.name = "DIA"

# Consumo acumulado de hoy (D a las 23:59) y rezagos de días anteriores (D-1, D-2 y D-7)
df_lags_frio["frio_hoy_d"] = serie_frio_cal
df_lags_frio["frio_lag_1"] = serie_frio_cal.shift(1)
df_lags_frio["frio_lag_2"] = serie_frio_cal.shift(2)
df_lags_frio["frio_lag_7"] = serie_frio_cal.shift(7)

# Cambio diario respecto a ayer (captura si la demanda térmica viene subiendo o bajando)
df_lags_frio["frio_diff_1"] = df_lags_frio["frio_hoy_d"] - df_lags_frio["frio_lag_1"]

# Promedios móviles (últimos 3, 7 y 14 días calendario) y desvío móvil de 7 días (estabilidad)
df_lags_frio["frio_roll_mean_3"] = serie_frio_cal.rolling(window=3, min_periods=2).mean()
df_lags_frio["frio_roll_mean_7"] = serie_frio_cal.rolling(window=7, min_periods=4).mean()
df_lags_frio["frio_roll_mean_14"] = serie_frio_cal.rolling(window=14, min_periods=7).mean()
df_lags_frio["frio_roll_std_7"] = serie_frio_cal.rolling(window=7, min_periods=4).std()

df_lags_frio = df_lags_frio.reset_index()

# Unimos en df_fe_parcial nuestras dos familias de features (Autorregresivas + Clima + Mes)
# Nota (Fase 1): se excluyen dia_semana y es_fin_de_semana porque el ANOVA demostró
# efecto semanal despreciable (eta^2 = 0.55%) frente a mes_numero (59.07%).
cols_clima_cand = [
    "DIA",
    "radiacion_roll7",
    "radiacion_mj_m2",
    "temp_aparente_c",
    "temp_media_roll7",
    "temp_min_c",
    "temp_media_c",
]

df_fe_parcial = (
    df_diario[["DIA", "Frio_manana_y"]]
    .merge(df_lags_frio, on="DIA", how="left")
    .merge(df_clima_full[cols_clima_cand], on="DIA", how="left")
)
df_fe_parcial["mes_numero"] = df_fe_parcial["DIA"].dt.month

# Evaluamos correlación con el target (D+1) SOLO de las nuevas variables creadas en este bloque
cols_autorreg_y_cal = [
    "frio_hoy_d",
    "frio_lag_1",
    "frio_lag_2",
    "frio_lag_7", #sirve para ver si el consumo es igual al de la semana pasada en ese dia.
    "frio_diff_1", #explica la velocidad y direccion del cambio en las ultimas 24hs
    "frio_roll_mean_3",
    "frio_roll_mean_7",
    "frio_roll_mean_14",
    "frio_roll_std_7",
    "mes_numero",
]

tabla_lags = pd.DataFrame({
    "r_Pearson_Target(D+1)": [df_fe_parcial[c].corr(df_fe_parcial["Frio_manana_y"]) for c in cols_autorreg_y_cal],
    "r_Spearman_Target(D+1)": [df_fe_parcial[c].corr(df_fe_parcial["Frio_manana_y"], method="spearman") for c in cols_autorreg_y_cal],
    "Nulos_en_1172_dias": [df_fe_parcial[c].isna().sum() for c in cols_autorreg_y_cal],
}, index=cols_autorreg_y_cal).sort_values("r_Pearson_Target(D+1)", key=abs, ascending=False).round(3)

print(f"Tabla unificada df_fe_parcial lista: {df_fe_parcial.shape[0]} días x {df_fe_parcial.shape[1]} columnas")
print("CORRELACIÓN DE VARIABLES AUTORREGRESIVAS Y CALENDARIO CON EL TARGET (D+1):")
print(tabla_lags.to_string())

Tabla unificada df_fe_parcial lista: 1172 días x 18 columnas
CORRELACIÓN DE VARIABLES AUTORREGRESIVAS Y CALENDARIO CON EL TARGET (D+1):
                   r_Pearson_Target(D+1)  r_Spearman_Target(D+1)  Nulos_en_1172_dias
frio_roll_mean_7                   0.843                   0.847                   3
frio_roll_mean_14                  0.842                   0.850                   6
frio_hoy_d                         0.841                   0.845                   0
frio_roll_mean_3                   0.828                   0.833                   7
frio_lag_1                         0.744                   0.754                  32
frio_lag_7                         0.736                   0.749                  46
frio_lag_2                         0.725                   0.739                  39
frio_roll_std_7                    0.178                   0.168                   3
frio_diff_1                        0.163                   0.166                  32
mes_numero    

Con la salida de df_fe_parcial se confirman unas conclusiones claves:
- Dominio de las medias móviles (0.842–0.843) sobre los rezagos puntuales (0.725–0.744): frio_roll_mean_7 (r = 0.843) y frio_roll_mean_14 (r = 0.842) explican mejor el consumo de mañana (D+1) que el valor aislado de hoy (frio_hoy_d, 0.841) y mucho mejor que mirar un día suelto de atrás (frio_lag_1, frio_lag_2, frio_lag_7).
- El cambio brusco diario (frio_diff_1, 0.163) y la volatilidad semanal (frio_roll_std_7, 0.178) tienen relación débil: sirven como señal complementaria menor, pero no marcan la tendencia principal.
- Por qué mes_numero marca r = -0.038 en Pearson?: en la Fase 1 vimos por ANOVA que el mes explica el 59.07% de la varianza, pero esa relación es en forma de campana (pico en junio–agosto, mínimo en diciembre–enero). Como Pearson mide solo rectas crecientes o decrecientes de 1 a 12, los extremos se cancelan; en cambio, la estacionalidad queda capturada de forma lineal por la radiación solar (radiacion_roll7, r = 0.742).
- Al haber alta redundancia entre frio_hoy_d, frio_roll_mean_3, frio_roll_mean_7 y frio_roll_mean_14, la Sección 2.3 (Selección de Variables) va a definir cuáles conviene conservar.


## Auditoria de reproducibilidad, tabla de resuemn de la seccion 2.2A y respaldo intermedio

In [5]:
import json
from src.auxiliar_functions import DATA_DIR, SEMILLA, huella, verificar_checksums

# Verificación reproducible de semilla, Excel crudos y dataset_trabajo.csv
assert SEMILLA == 42, f"Se esperaba SEMILLA=42 y se obtuvo {SEMILLA}"
verificar_checksums()

with open(DATA_DIR / "checksums.json", "r", encoding="utf-8") as f:
    registro_checksums = json.load(f)

hash_trabajo_actual = huella(PROCESSED_DATA_DIR / "dataset_trabajo.csv")
assert hash_trabajo_actual == registro_checksums["data/processed/dataset_trabajo.csv"], (
    "El hash de dataset_trabajo.csv no coincide con data/checksums.json"
)

# 3. Tabla resumen consolidada de las 16 variables creadas en la Sección 2.2A
catalogo_22a = {
    "frio_roll_mean_14": ("Autorregresiva (Frío)",  "Evaluar colinealidad vs roll_7 en 2.3"),
    "frio_roll_mean_7":  ("Autorregresiva (Frío)",  "Principal (inercia térmica semanal)"),
    "frio_hoy_d":        ("Autorregresiva (Frío)",  "Principal (nivel base día D a las 23:59)"),
    "frio_roll_mean_3":  ("Autorregresiva (Frío)",  "Evaluar colinealidad vs roll_7 en 2.3"),
    "frio_lag_1":        ("Autorregresiva (Frío)",  "Evaluar en 2.3 (32 nulos en huecos)"),
    "frio_lag_7":        ("Autorregresiva (Frío)",  "Revisar/posible descarte en 2.3 (46 nulos)"),
    "radiacion_roll7":   ("Clima Tecate",           "Principal (mejor predictor climático)"),
    "frio_lag_2":        ("Autorregresiva (Frío)",  "Revisar/posible descarte en 2.3 (39 nulos)"),
    "radiacion_mj_m2":   ("Clima Tecate",           "Principal (radiación diaria)"),
    "temp_aparente_c":   ("Clima Tecate",           "Principal (sensación térmica)"),
    "temp_media_roll7":  ("Clima Tecate",           "Complementaria (inercia térmica aire)"),
    "temp_min_c":        ("Clima Tecate",           "Principal (carga térmica nocturna)"),
    "temp_media_c":      ("Clima Tecate",           "Evaluar colinealidad vs aparente en 2.3"),
    "frio_roll_std_7":   ("Autorregresiva (Frío)",  "Complementaria (volatilidad semanal)"),
    "frio_diff_1":       ("Autorregresiva (Frío)",  "Complementaria (cambio 24h)"),
    "mes_numero":        ("Calendario",             "No lineal (útil en árboles; baja r lineal)"),
}

filas_tabla = []
for col, (familia, rol) in catalogo_22a.items():
    r_p = df_fe_parcial[col].corr(df_fe_parcial["Frio_manana_y"], method="pearson")
    r_s = df_fe_parcial[col].corr(df_fe_parcial["Frio_manana_y"], method="spearman")
    nulos = int(df_fe_parcial[col].isna().sum())
    filas_tabla.append({
        "Variable": col,
        "Familia": familia,
        "r_Pearson(D+1)": round(r_p, 3),
        "r_Spearman(D+1)": round(r_s, 3),
        "Nulos": nulos,
        "Completitud_%": round(100 * (1 - nulos / len(df_fe_parcial)), 1),
        "Estado_hacia_2_3": rol,
    })

tabla_consolidada_22a = (
    pd.DataFrame(filas_tabla)
    .sort_values("r_Spearman(D+1)", key=abs, ascending=False)
    .set_index("Variable")
)

print(f"Reproducibilidad OK | SEMILLA = {SEMILLA} | SHA-256 dataset_trabajo = {hash_trabajo_actual[:16]}...")
print(f"Variables en memoria (df_fe_parcial): {df_fe_parcial.shape[0]} días x {df_fe_parcial.shape[1]} columnas")
print(f"Días con Target(D+1) válido: {df_fe_parcial['Frio_manana_y'].notna().sum()} / {len(df_fe_parcial)}\n")
print("TABLA RESUMEN CONSOLIDADA — VARIABLES CREADAS EN SECCIÓN 2.2A:")
print(tabla_consolidada_22a.to_string())

Verificación de integridad completada: 3 archivo(s) coinciden.
Reproducibilidad OK | SEMILLA = 42 | SHA-256 dataset_trabajo = bd015abb722ba215...
Variables en memoria (df_fe_parcial): 1172 días x 18 columnas
Días con Target(D+1) válido: 1140 / 1172

TABLA RESUMEN CONSOLIDADA — VARIABLES CREADAS EN SECCIÓN 2.2A:
                                 Familia  r_Pearson(D+1)  r_Spearman(D+1)  Nulos  Completitud_%                            Estado_hacia_2_3
Variable                                                                                                                                   
frio_roll_mean_14  Autorregresiva (Frío)           0.842            0.850      6           99.5       Evaluar colinealidad vs roll_7 en 2.3
frio_roll_mean_7   Autorregresiva (Frío)           0.843            0.847      3           99.7         Principal (inercia térmica semanal)
frio_hoy_d         Autorregresiva (Frío)           0.841            0.845      0          100.0    Principal (nivel base día D 